# Compression-robust, calibrated deepfake detection (Kaggle)

Setup: Accelerator GPU T4, Internet on. Add the *Deepfake Detection Challenge* competition data as input, and upload this repository as a Dataset named `deepfake-code`.

Run the cells in order, starting with the smoke test.

In [ ]:
import os, shutil, subprocess, sys
SRC = [p for p in ['/kaggle/input/deepfake-code/deepfake-detector', '/kaggle/input/deepfake-code'] if os.path.exists(os.path.join(p, 'v2'))][0]
WORK = '/kaggle/working/deepfake-detector'
if os.path.exists(WORK): shutil.rmtree(WORK)
shutil.copytree(SRC, WORK)
os.chdir(WORK)
print('code at', WORK, os.listdir('.'))

In [ ]:
!pip install -q "opencv-python-headless>=4.9,<5" scikit-learn matplotlib joblib
import sklearn, cv2; print("sklearn", sklearn.__version__, "| cv2", cv2.__version__)
print("Serving requirements must pin scikit-learn==" + sklearn.__version__)

In [ ]:
DFDC = '/kaggle/input/deepfake-detection-challenge/train_sample_videos'
!PYTHONPATH=. python -m v2.make_manifest --dfdc_dir {DFDC} --out manifest.csv
!head -3 manifest.csv

## Smoke test (20 videos). The output should report `detector=yunet`.

In [ ]:
!PYTHONPATH=. python -m v2.extract --manifest manifest.csv --out smoke --level orig --limit 20

## Feature extraction
One run per compression level over the same videos. `--severity` only orders the plot's x-axis.

In [ ]:
LIMIT = ''
!PYTHONPATH=. python -m v2.extract --manifest manifest.csv --out feats --level orig --severity 0 --workers 1 {LIMIT}
!PYTHONPATH=. python -m v2.extract --manifest manifest.csv --out feats --level crf28 --crf 28 --severity 28 --workers 1 {LIMIT}
!PYTHONPATH=. python -m v2.extract --manifest manifest.csv --out feats --level crf35 --crf 35 --severity 35 --workers 1 {LIMIT}
!PYTHONPATH=. python -m v2.extract --manifest manifest.csv --out feats --level social --scale 480 --crf 32 --severity 38 --workers 1 {LIMIT}
!PYTHONPATH=. python -m v2.extract --manifest manifest.csv --out feats --level crf42 --crf 42 --severity 42 --workers 1 {LIMIT}

## Train, evaluate and save the model

In [ ]:
!PYTHONPATH=. python -m v2.evaluate --features_dir feats --out_dir results --seeds 5 --save_model results/model.joblib

In [ ]:
from IPython.display import Image, display, Markdown
display(Markdown(open('results/results.md').read()))
display(Image('results/compression.png')); display(Image('results/risk_coverage.png'))

In [ ]:
!cd results && zip -q -r ../results.zip results.md results.json compression.png risk_coverage.png model.joblib && ls -la ../results.zip